In [ ]:
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import zarr
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
import geopandas as gpd
import spatialdata as sd
import spatialdata_plot
from spatialdata.models import PointsModel, ShapesModel
from spatialdata.transformations import Affine, Sequence, get_transformation, set_transformation
from anndata.io import read_elem
from scipy.cluster.hierarchy import leaves_list, linkage
import decoupler as dc

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

BASE = Path.cwd()
RESULTS = BASE / "results"
OUT_DIR = RESULTS / "06_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
PANELS = {
    "1b": (50, 45),
    "1c": (45, 45),
    "1d-f": (180, 64),
    "1g-i": (180, 59),
    "2a": (180, 61),
    "2b": (68, 52),
    "2c": (68, 52),
    "2d": (47, 52),
    "2e-f": (180, 49.5),
    "1Sa": (110, 60),
    "1Sb": (70, 40),
    "2Sa": (180, 46),
    "2Sb": (78, 60),
    "2Sc": (102, 60),
    "2Sd": (134.5, 62.5),
    "2Se": (44.5, 62.5),
}
SUPP_PT = 7
fs.DPI = 500
plt.rcParams.update({
    "figure.dpi": fs.DPI,
    "font.size": 6, "axes.titlesize": 6, "axes.labelsize": 6, "figure.titlesize": 6, "legend.title_fontsize": 6,
    "xtick.labelsize": 5.5, "ytick.labelsize": 5.5, "legend.fontsize": 5.5,
})

def panel_size(panel):
    width, height = PANELS[panel]
    return width * fs.MM, height * fs.MM

def save_panel(fig, name, tight=False):
    with plt.rc_context({"savefig.bbox": "tight" if tight else "standard", "savefig.pad_inches": 0}):
        fs.save(fig, OUT_DIR / name)

def font7(fig):
    for ax in fig.axes:
        ax.tick_params(labelsize=SUPP_PT)
    for text in fig.findobj(mpl.text.Text):
        text.set_fontsize(SUPP_PT)
    return fig

def fit_panel(draw, panel):
    target = np.array(panel_size(panel))
    size = target.copy()
    for _ in range(3):
        fig = draw(*size)
        size = size - (fig.get_tightbbox(fig.canvas.get_renderer()).size - target)
        plt.close(fig)
    return draw(*size)

In [ ]:
annotated = zarr.open_group(BASE / "data" / "integrated_mouse_annotated.zarr", mode="r")
obs = read_elem(annotated["obs"])
umap = np.asarray(read_elem(annotated["obsm/X_umap"]))
spatial = np.asarray(read_elem(annotated["obsm/spatial"]))
SECTIONS = list(obs["section"].cat.categories)

GROUPS = ["C-ion", "γ-ray"]
ROWS = [[section for section in SECTIONS if section.startswith(radiation)] for radiation in GROUPS]
ROI_LABEL = {section: "ROI" + section.split("ROI ")[1].rstrip(")") for section in SECTIONS}
TIMEPOINT = {section: section.split(" ", 1)[1].split(" (")[0] for section in SECTIONS}

In [ ]:
fig, ax = plt.subplots(figsize=panel_size("1b"), layout="constrained")
labels = pd.Categorical(obs["cell_largetype"]).remove_unused_categories()
cats = labels.categories
order = np.random.default_rng(0).permutation(len(umap))
colors = np.array(fs.palette(labels.astype(str)))
ax.scatter(umap[order, 0], umap[order, 1], c=colors[order], s=0.2, lw=0, rasterized=True)
ax.set_aspect("equal")
ax.set_axis_off()

handles = [Line2D([], [], ls="", marker="o", ms=3, mew=0, color=c) for c in fs.palette(cats)]
fig.legend(handles, cats, loc="outside lower center", ncol=3, columnspacing=0.6, handletextpad=0.1, labelspacing=0.25)

span = np.ptp(umap, axis=0).max()
x0, y0 = umap.min(axis=0) - 0.03 * span
arrow = dict(arrowstyle="-|>", lw=0.5, color=fs.BLACK, mutation_scale=4, shrinkA=0, shrinkB=0)
ax.update_datalim([(x0, y0)])
ax.annotate("", (x0 + 0.15 * span, y0), (x0, y0), arrowprops=arrow)
ax.annotate("", (x0, y0 + 0.15 * span), (x0, y0), arrowprops=arrow)
ax.annotate("UMAP1", (x0 + 0.075 * span, y0), xytext=(0, -2), textcoords="offset points", fontsize=5, ha="center", va="top")
ax.annotate("UMAP2", (x0, y0 + 0.075 * span), xytext=(-2, 0), textcoords="offset points", fontsize=5,
            ha="right", va="center", rotation=90)
fig.get_layout_engine().set(w_pad=0.01, h_pad=0.01)
save_panel(fig, "1b_umap")
plt.show()

In [ ]:
fibroblasts = obs[obs["cell_largetype"] == "Fibroblast"]
with plt.rc_context({"xtick.labelsize": 5, "ytick.labelsize": 5, "legend.fontsize": 6}):
    ct = pd.crosstab(fibroblasts["section"], fibroblasts["cell_subtype"]).reindex(index=SECTIONS, fill_value=0)
    ct = ct.loc[:, ct.sum() > 0]
    colors = [fs.CELL_COLORS.get(c, fs.LIGHTGREY) for c in ct.columns]
    fig, axes = plt.subplots(len(ROWS), 1, figsize=panel_size("1c"), layout="constrained", squeeze=False, sharey="col")
    for r, row in enumerate(ROWS):
        ax = axes[r, 0]
        table = ct.loc[row] / 1000
        xs = np.arange(len(table))
        bottom = np.zeros(len(table))
        for c, color in zip(table.columns, colors):
            ax.bar(xs, table[c], bottom=bottom, width=0.75, color=color, edgecolor="white", linewidth=0.25, label=c)
            bottom += table[c].to_numpy(dtype=float)
        ax.set_xticks(xs, [ROI_LABEL[g] for g in table.index])
        ax.tick_params(axis="x", length=0, pad=1)
        ax.tick_params(axis="y", pad=1)
        ax.set_xlim(-0.5, len(table) - 0.5)
        times = [TIMEPOINT[g] for g in table.index]
        start = 0
        for i in range(1, len(times) + 1):
            if i == len(times) or times[i] != times[start]:
                ax.annotate(times[start], ((start + i - 1) / 2, 0), xycoords=("data", "axes fraction"),
                            xytext=(0, -(2 + mpl.rcParams["xtick.labelsize"])), textcoords="offset points", ha="center", va="top",
                            fontsize=mpl.rcParams["axes.labelsize"])
                start = i
        ax.text(0.02, 1.0, GROUPS[r], transform=ax.transAxes, ha="left", va="top")
    fig.supylabel("Number of cells (×10$^3$)", fontsize=mpl.rcParams["axes.labelsize"])
    handles, names = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, names, loc="outside lower center", ncol=2, columnspacing=0.8)
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.01, hspace=0.04)
save_panel(fig, "1c_fibroblast_count")
plt.show()

In [ ]:
XENIUM_ZARR = BASE.parent / "xenium_data_mouse" / "integrated" / "integrated_mouse.zarr"
FRAMES = {
    "γ-ray Sham (ROI 8)": ((4310, 16140), 5, 0),
    "C-ion 3.5 d (ROI 6)": ((1810, 15440), 30, 90),
    "γ-ray 3.5 d (ROI 13)": ((7800, 14400), 350, 0),
}
INSETS = {"C-ion 3.5 d (ROI 6)": (1750, 15730), "γ-ray 3.5 d (ROI 13)": (7980, 14630)}
FRAME_SIZE, INSET_SIZE, SCALE_BAR = 700, 200, 50
CHANNEL_COLORS = {"DAPI": "#3D7BFF", "ATP1A1/CD45/E-Cadherin": "#FF00FF", "alphaSMA/Vimentin": "#00FF00"}
PERCENTILE = (1, 99)
BRIGHTNESS = 0.5
DOWNSAMPLE = 8
FIBROBLASTS = ["Telocyte", "Trophocyte", "Adamdec1+ fibroblast", "Villus fibroblast"]
MMP_CELLS = ["Adamdec1+ fibroblast", "Macrophage"]
GENE_COLORS = {"Mmp12": "#FFE600", "Mmp13": "#FF2A2A"}
QV = 20

warnings.filterwarnings("ignore", message="SpatialData is not stored in the most current format")
warnings.filterwarnings("ignore", message="The object has `points` element")
sdata = sd.read_zarr(XENIUM_ZARR, selection=("images", "shapes", "points"))
boundary = sdata["cell_boundaries"]
transcripts = sdata["transcripts"]

def rotation(center, deg):
    (cx, cy), t = center, np.deg2rad(deg)
    c, s = np.cos(t), np.sin(t)
    return Affine([[c, -s, cx - c * cx + s * cy], [s, c, cy - s * cx - c * cy], [0, 0, 1]], input_axes=("x", "y"), output_axes=("x", "y"))

sections, pixels = {}, {channel: [] for channel in CHANNEL_COLORS}
for section, (center, angle, turn) in FRAMES.items():
    cells_sec = obs[obs["section"] == section]
    roi_id = f"{cells_sec['condition'].iloc[0]}_{cells_sec['slide'].iloc[0]}"
    image = sdata[f"morphology_focus_{roi_id}"]
    small = image.coarsen(y=DOWNSAMPLE, x=DOWNSAMPLE, boundary="trim").mean().compute()
    for channel in CHANNEL_COLORS:
        values = small.sel(c=channel).values.ravel()
        pixels[channel].append(values[values > 0])

    shapes = {}
    for name, cell_types in [("fibroblasts", FIBROBLASTS), ("mmp_cells", MMP_CELLS)]:
        shown = cells_sec[cells_sec["cell_subtype"].isin(cell_types)]
        shapes[name] = ShapesModel.parse(gpd.GeoDataFrame(shown[["cell_subtype"]], geometry=boundary.geometry.loc[shown.index].values),
                                         transformations={"global": get_transformation(boundary, "global")})
    genes = transcripts[(transcripts["roi_id"] == roi_id) & transcripts["feature_name"].isin(list(GENE_COLORS))
                        & (transcripts["qv"] >= QV)][["x", "y", "feature_name"]].compute()
    genes["feature_name"] = pd.Categorical(genes["feature_name"].astype(str), categories=list(GENE_COLORS))
    points = PointsModel.parse(genes.reset_index(drop=True), coordinates={"x": "x", "y": "y"},
                               transformations={"global": get_transformation(transcripts, "global")})

    sec = sections[section] = sd.SpatialData(images={"image": image}, shapes=shapes, points={"transcripts": points})
    for name in ["image", *shapes, "transcripts"]:
        set_transformation(sec[name], Sequence([get_transformation(sec[name], "global"), rotation(center, angle - turn)]), "zoom")
limits = {channel: np.percentile(np.concatenate(values), PERCENTILE) for channel, values in pixels.items()}

def zoom_panel(ax, section, center, size, shapes, colors, label, point_size=None):
    half = size / 2
    cropped = sections[section].filter_by_coordinate_system("zoom").query.bounding_box(
        axes=["x", "y"], min_coordinate=[center[0] - half, center[1] - half], max_coordinate=[center[0] + half, center[1] + half],
        target_coordinate_system="zoom")
    plot = (cropped.pl.render_images("image", channel=list(CHANNEL_COLORS), palette=list(CHANNEL_COLORS.values()),
                                     norm=[Normalize(*limits[channel]) for channel in CHANNEL_COLORS], scale="full", alpha=BRIGHTNESS)
            .pl.render_shapes(shapes, color="cell_subtype", palette=colors, fill_alpha=1, outline_alpha=1,
                              outline_color="black", outline_width=0.1))
    if point_size:
        plot = plot.pl.render_points("transcripts", color="feature_name", palette=GENE_COLORS, size=point_size, method="matplotlib")
    plot.pl.show(coordinate_systems="zoom", ax=ax, legend_loc=None, colorbar=False)
    ax.set_facecolor("black")
    ax.set(xlim=(center[0] - half, center[0] + half), ylim=(center[1] + half, center[1] - half), aspect="equal",
           xticks=[], yticks=[], ylabel="", title="")
    ax.set_xlabel(label, labelpad=2, fontsize=7)
    ax.spines[:].set_visible(False)
    x, y = center[0] - half + 0.05 * size, center[1] + half - 0.04 * size
    ax.plot([x, x + SCALE_BAR], [y, y], color="white", lw=1.0, solid_capstyle="butt")
    ax.annotate(f"{SCALE_BAR:g} µm", (x + SCALE_BAR / 2, y), xytext=(0, 2), textcoords="offset points",
                ha="center", va="bottom", fontsize=6, color="white")

In [ ]:
fibroblast_colors = dict(zip(FIBROBLASTS, fs.palette(FIBROBLASTS)))
fig, axes = plt.subplots(1, len(FRAMES), figsize=panel_size("1d-f"), layout="constrained")
for ax, (section, (center, _, _)) in zip(axes, FRAMES.items()):
    zoom_panel(ax, section, center, FRAME_SIZE, "fibroblasts", fibroblast_colors, section)
fig.legend(handles=[Patch(fc=color, ec="black", lw=0.3, label=name) for name, color in fibroblast_colors.items()],
           loc="outside lower center", ncol=len(FIBROBLASTS), frameon=False, handlelength=1, handleheight=0.8, fontsize=6, borderaxespad=0.2)
fig.get_layout_engine().set(h_pad=0.01)
save_panel(fig, "1d-f_zoom")
plt.show()

In [ ]:
de_results = pd.read_csv(RESULTS / "03_fibroblast_gene_expression_analysis" / "fibroblast_deseq2_vs_sham.csv")

FC_CUTOFF = 0.5

lfc = de_results.pivot_table(index=["cell", "gene"], columns=["radiation", "time"], values="log2FoldChange")
AXIS_LIM = np.nanquantile(lfc, [0.005, 0.995])

GROUP_COLORS = {
    "Other": (fs.LIGHTGREY, 2),
    "Up with C-ion only": (fs.CION, 3),
    "Up with γ-ray only": (fs.GAMMA, 3),
    "Up with both": (fs.BOTH, 3),
}

def classify(cell, time):
    x = lfc.loc[cell, ("C-ion", time)]
    y = lfc.loc[cell, ("γ-ray", time)]
    group = pd.Series("Other", index=x.index)
    group[(x >= FC_CUTOFF) & (y < FC_CUTOFF)] = "Up with C-ion only"
    group[(x < FC_CUTOFF) & (y >= FC_CUTOFF)] = "Up with γ-ray only"
    group[(x >= FC_CUTOFF) & (y >= FC_CUTOFF)] = "Up with both"
    return group

def scatter_panel(ax, cell, time):
    x = lfc.loc[cell, ("C-ion", time)]
    y = lfc.loc[cell, ("γ-ray", time)]
    group = classify(cell, time)
    for name, (color, size) in GROUP_COLORS.items():
        in_group = group == name
        ax.scatter(x[in_group].clip(AXIS_LIM[0], AXIS_LIM[1]), y[in_group].clip(AXIS_LIM[0], AXIS_LIM[1]),
                   s=size, color=color, lw=0, rasterized=True, label=name)

    ax.plot(AXIS_LIM, AXIS_LIM, color=fs.GREY, lw=0.4, ls="--")
    ax.axvline(FC_CUTOFF, color=fs.GREY, lw=0.4, ls=":")
    ax.axhline(FC_CUTOFF, color=fs.GREY, lw=0.4, ls=":")
    ax.set_xlim(AXIS_LIM)
    ax.set_ylim(AXIS_LIM)
    ax.set_aspect("equal")
    ax.set_title(f"{cell}, {time}")
    ax.set_xlabel("C-ion: log$_2$ fold change vs Sham")
    ax.set_ylabel("γ-ray: log$_2$ fold change vs Sham")

fig, axes = plt.subplots(1, 3, figsize=panel_size("1g-i"), layout="constrained")
for ax, time in zip(axes, ["24 h", "48 h", "3.5 d"]):
    scatter_panel(ax, "Adamdec1+ fibroblast", time)
fig.legend(handles=[Line2D([], [], ls="", marker="o", ms=3, mew=0, color=color, label=name) for name, (color, _) in GROUP_COLORS.items()],
           loc="outside lower center", ncol=len(GROUP_COLORS), borderaxespad=0.2)
fig.get_layout_engine().set(h_pad=0.01)
save_panel(fig, "1g-i_fourway")
plt.show()

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_mouse_annotated.zarr")

In [ ]:
obs_before = pd.read_parquet(RESULTS / "00_preprocess_and_cluster" / "qc_per_cell_before.parquet")
ROIS = pd.read_csv(RESULTS / "00_preprocess_and_cluster" / "roi_numbers.csv", index_col="roi_no")
MIN_COUNTS, MIN_GENES = 10, 5

def qc_violin(ax, values, labels, cutoff, ylabel, colors):
    data = [np.log10(np.asarray(v, dtype=float) + 1) for v in values]
    xs = np.arange(1, len(data) + 1)
    parts = ax.violinplot(data, positions=xs, widths=0.8, showextrema=False)
    for body, color in zip(parts["bodies"], colors):
        body.set(facecolor=color, edgecolor=fs.BLACK, linewidth=0.3, alpha=1)
    ax.hlines([np.median(d) for d in data], xs - 0.25, xs + 0.25, color=fs.BLACK, lw=0.75)
    ax.axhline(np.log10(cutoff + 1), color=fs.GREY, lw=0.6, ls=(0, (3, 2)))
    top = max(d.max() for d in data)
    ticks = [t for t in [0, 10, 100, 1000, 10000, 100000] if np.log10(t + 1) <= top * 1.02]
    ax.set_yticks(np.log10(np.array(ticks) + 1), [f"{t:,}" for t in ticks])
    ax.set_xticks(xs, labels)
    ax.set_xlim(0.4, len(data) + 0.6)
    ax.set_ylabel(ylabel)

fig, (ax_counts, ax_genes) = plt.subplots(2, 1, figsize=panel_size("1Sa"), sharex=True, layout="constrained")
fig.get_layout_engine().set(h_pad=0.5 * fs.MM, hspace=0)
labels = list(ROIS["section"])
colors = [fs.ROI_COLORS[c] for c in ROIS["condition"]]

counts, genes = [], []
for roi in ROIS.index:
    roi_cells = obs_before[obs_before["roi_no"].astype(int) == roi]
    counts.append(roi_cells["total_counts"])
    genes.append(roi_cells["n_genes_by_counts"])

qc_violin(ax_counts, counts, labels, MIN_COUNTS, "Transcripts" + chr(10) + "per cell", colors)
qc_violin(ax_genes, genes, labels, MIN_GENES, "Genes" + chr(10) + "per cell", colors)
plt.setp(ax_genes.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
fig.align_ylabels([ax_counts, ax_genes])
font7(fig)
save_panel(fig, "1Sa_qc_violin")
plt.show()

In [ ]:
FIBROBLAST_MARKERS = {
    "Telocyte":             ["Pdgfra", "Bmp4"],
    "Trophocyte":           ["Grem1", "Rspo3"],
    "Adamdec1+ fibroblast": ["Adamdec1"],
    "Villus fibroblast":    ["Fgfr2", "Col14a1"],
}

def tidy_dotplot(dp):
    axes = dp.get_axes()
    size_ax, color_ax = axes["size_legend_ax"], axes["color_legend_ax"]
    size_ax.set_xticklabels([t.get_text() if i % 2 == 0 else "" for i, t in enumerate(size_ax.get_xticklabels())])
    for dots in size_ax.collections:
        dots.set_sizes([v if i % 2 == 0 else 0 for i, v in enumerate(dots.get_sizes())])
    fig_w, fig_h = dp.fig.get_size_inches()
    c, s = color_ax.get_position(), size_ax.get_position()
    color_ax.set_position([c.x0, c.y0, c.width, 0.08 / fig_h])
    size_ax.set_position([s.x0, c.y0 + (0.08 + 0.40) / fig_h, s.width, 0.12 / fig_h])
    size_ax.set_ylim(-0.6, 0.6)
    size_ax.tick_params(axis="x", length=0, pad=1)
    size_ax.set_title(size_ax.get_title(), y=1.0, pad=2, size="small")
    right = min(s.x0, c.x0) - 0.15 / fig_w
    for k in ["mainplot_ax", "gene_group_ax"]:
        if k in axes:
            p = axes[k].get_position()
            axes[k].set_position([p.x0, p.y0, right - p.x0, p.height])
    ax = axes["mainplot_ax"]
    for t in ax.get_xticklabels():
        t.set_fontstyle("italic")
    return ax

fibroblasts = adata[adata.obs["cell_subtype"].isin(FIBROBLAST_MARKERS)].copy()
fibroblasts.obs["cell_subtype"] = pd.Categorical(fibroblasts.obs["cell_subtype"].astype(str), categories=list(FIBROBLAST_MARKERS))
def draw(width, height):
    dp = sc.pl.dotplot(fibroblasts, sum(FIBROBLAST_MARKERS.values(), []), groupby="cell_subtype", layer="log1p_norm",
                       standard_scale="var", dot_min=0, dot_max=1, figsize=(width, height), show=False, return_fig=True)
    dp.style(largest_dot=40, dot_edge_lw=0.2)
    dp.legend(width=0.7, size_title="% cells", colorbar_title="Scaled\nexpression")
    dp.make_figure()
    tidy_dotplot(dp).set_ylabel("Cell type")
    return font7(dp.fig)

fig = fit_panel(draw, "1Sb")
save_panel(fig, "1Sb_fibroblast_dotplot", tight=True)
plt.show()

In [ ]:
cells = pd.read_parquet(BASE / "data" / "cn_schurch.parquet")
cells[["x", "y"]] = spatial[cells["index"].to_numpy()]
ROTATION = {
    "C-ion Sham (ROI 1)": 227, "C-ion 24 h (ROI 2)": 302, "C-ion 24 h (ROI 3)": 177, "C-ion 48 h (ROI 4)": 65,
    "C-ion 48 h (ROI 5)": 122, "C-ion 3.5 d (ROI 6)": 99, "C-ion 3.5 d (ROI 7)": 140,
    "γ-ray Sham (ROI 8)": 297, "γ-ray 24 h (ROI 9)": 15, "γ-ray 24 h (ROI 10)": 32, "γ-ray 48 h (ROI 11)": 5,
    "γ-ray 48 h (ROI 12)": 290, "γ-ray 3.5 d (ROI 13)": 252, "γ-ray 3.5 d (ROI 14)": 55,
}
for section, degrees in ROTATION.items():
    roi = cells["section"] == section
    xy = cells.loc[roi, ["x", "y"]].to_numpy()
    t = np.radians(degrees)
    cells.loc[roi, ["x", "y"]] = (xy - xy.mean(axis=0)) @ np.array([[np.cos(t), np.sin(t)], [-np.sin(t), np.cos(t)]])
cn_info = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "cn_info.csv")

slide_of = obs.groupby("section", observed=True)["slide"].first()
FIG2_SECTIONS = [section for section in SECTIONS if slide_of[section] == "No1" and section != "C-ion Sham (ROI 1)"]
FIG2_GRID = [[None] + [s for s in FIG2_SECTIONS if s.startswith("C-ion")],
             [s for s in FIG2_SECTIONS if s.startswith("γ-ray")]]

CN_COLORS = dict(zip([f"CN{cn} {name}" for cn, name in zip(cn_info["cn"], cn_info["name"])], [
    "#fdd0a2", "#fdae6b", "#fd8d3c", "#d94801", "#e7ba52", "#bd9e39", "#8c6d31",
    "#9ecae1", "#3182bd", "#08306b", "#e7298a", "#de2d26",
    "#a1d99b", "#238b45",
    "#636363",
]))

def fit_roi(ax):
    ax.margins(0)
    ax.set_aspect("equal", adjustable="datalim")
    ax.invert_yaxis()
    ax.axis("off")

def roi_map(column, colors, name, panel, title, grid=None, fontsize=5, title_fontsize=6, wrap=False):
    if grid is None:
        grid = [SECTIONS[:7], SECTIONS[7:]]
    fig, axes = plt.subplots(len(grid), len(grid[0]), figsize=panel_size(panel))
    legend = [plt.Line2D([], [], marker="o", ls="", ms=3, color=color, label=label) for label, color in colors.items()]
    compact = dict(labelspacing=0.12, handletextpad=0.1, borderaxespad=0, borderpad=0) if wrap else {}
    for ax, section in zip(axes.flat, sum(grid, [])):
        if section is None:
            ax.axis("off")
            ax.legend(handles=legend, loc="center", frameon=False, fontsize=fontsize, title=title, title_fontsize=title_fontsize)
            continue
        roi_cells = cells[cells["section"] == section]
        ax.scatter(roi_cells["x"], roi_cells["y"], c=roi_cells[column].map(colors).values, s=0.05, linewidths=0, rasterized=True)
        fit_roi(ax)
        label = section.replace(" (", chr(10) + "(") if wrap else section
        ax.text(0.5, -0.02, label, transform=ax.transAxes, ha="center", va="top", fontsize=fontsize, linespacing=1.0)
    if None in sum(grid, []):
        fig.subplots_adjust(left=0.005, right=0.995, top=0.995, bottom=0.07, wspace=0.03, hspace=0.16)
    else:
        fig.legend(handles=legend, loc="center right", frameon=False, fontsize=fontsize, title=title, title_fontsize=title_fontsize, **compact)
        if wrap:
            fig.subplots_adjust(left=0.005, right=0.74, top=0.99, bottom=0.13, wspace=0.06, hspace=0.32)
        else:
            fig.subplots_adjust(left=0.005, right=0.80 if len(colors) > 6 else 0.87, top=0.99, bottom=0.07, wspace=0.03, hspace=0.2)
    save_panel(fig, name)
    plt.show()

In [ ]:
roi_map("layer", fs.LAYER_COLORS, "2a_layer_map", "2a", "Layer", grid=FIG2_GRID, fontsize=7, title_fontsize=7)

In [ ]:
TIMES = ["Sham", "24 h", "48 h", "3.5 d"]

def stacked_by_time(counts, colors, panel, ylabel):
    radiation = counts.index.map(lambda section: section.split(" ")[0])
    pooled = counts.groupby([radiation, counts.index.map(TIMEPOINT)]).sum()
    frac = pooled.div(pooled.sum(axis=1), axis=0) * 100
    fig, axes = plt.subplots(len(GROUPS), 1, figsize=panel_size(panel), layout="constrained", squeeze=False, sharey="col")
    for ax, group in zip(axes[:, 0], GROUPS):
        table = frac.loc[group].reindex(TIMES)
        xs = np.arange(len(table))
        bottom = np.zeros(len(table))
        for name, color in colors.items():
            ax.bar(xs, table[name], bottom=bottom, width=0.75, color=color, edgecolor="white", linewidth=0.25, label=name)
            bottom += table[name].to_numpy(dtype=float)
        ax.set_xticks(xs, TIMES)
        ax.set_title(group)
        ax.set_ylim(0, 100)
    fig.supylabel(ylabel, fontsize=mpl.rcParams["axes.labelsize"])
    handles, names = axes[0, 0].get_legend_handles_labels()
    names = [name.replace("/", "/\n", 1) if "/" in name else "\n".join(name.rsplit(" ", 1)) if len(name) > 20 else name
             for name in names]
    fig.legend(handles[::-1], names[::-1], loc="outside right center", alignment="left", labelspacing=0.2)
    return fig

crypt_counts = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "layer_cn_composition_by_roi_counts.csv", index_col=[0, 1]).loc["Crypt"]
crypt_counts = crypt_counts.dropna(axis=1, how="all").fillna(0)
fig = stacked_by_time(crypt_counts, {k: v for k, v in CN_COLORS.items() if k in crypt_counts.columns}, "2b", "Crypt: proportion of cells (%)")
save_panel(fig, "2b_crypt_cn_by_time")
plt.show()

In [ ]:
cn9_counts = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "crypt_cn_composition_counts.csv", index_col=[0, 1]).loc["CN9"]
cn9_counts = cn9_counts.loc[:, cn9_counts.sum() > 0]
fig = stacked_by_time(cn9_counts, {celltype: fs.CELL_COLORS.get(celltype, fs.LIGHTGREY) for celltype in cn9_counts.columns},
                      "2c", "CN9: proportion of cells (%)")
save_panel(fig, "2c_cn9_celltype_by_time")
plt.show()

In [ ]:
niche_de = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "deseq2_niche_gamma_vs_cion.csv")
cn9 = niche_de[(niche_de["cn"] == "CN9") & (niche_de["time"] == "3.5 d")].set_index("gene")

results_df = cn9
sig = results_df[(results_df["padj"] < 0.05) & (results_df["log2FoldChange"].abs() > 0.5)]
top_genes = list(sig.nsmallest(12, "padj").index)
fig, ax = plt.subplots(figsize=panel_size("2d"), layout="constrained")
dc.pl.volcano(results_df.dropna(subset=["padj"]), x="log2FoldChange", y="padj",
              top=top_genes,
              color_pos=fs.UP, color_neg=fs.DOWN, color_null=fs.LIGHTGREY,
              kw_scatter={"s": 4, "lw": 0, "rasterized": True}, ax=ax)
for line in ax.lines:
    line.set(color=fs.GREY, lw=0.4, ls=(0, (2, 2)))
for text in ax.texts:
    text.set(fontsize=7, fontstyle="italic")
ax.set_xlabel("log$_2$ fold change")
ax.set_ylabel("−log$_{10}$(adjusted $\\it{P}$ value)")
ax.set_title("CN9 Adamdec1+ fibroblast-rich", fontsize=6)
fig.legend(handles=[Line2D([], [], ls="", marker="o", ms=3, mew=0, color=fs.UP, label=f"Upregulated in {fs.label('gamma')} at 3.5 d"),
                    Line2D([], [], ls="", marker="o", ms=3, mew=0, color=fs.DOWN, label=f"Upregulated in {fs.label('Cion')} at 3.5 d")],
           loc="outside lower center", ncol=1, handletextpad=0.2, labelspacing=0.2)
save_panel(fig, "2d_cn9_volcano_3.5d")
plt.show()

In [ ]:
mmp_colors = dict(zip(MMP_CELLS, fs.palette(MMP_CELLS)))
fig = plt.figure(figsize=panel_size("2e-f"), layout="constrained")
for subfig, (section, inset) in zip(fig.subfigures(1, len(INSETS)), INSETS.items()):
    ax_zoom, ax_inset = subfig.subplots(1, 2)
    center, angle, turn = FRAMES[section]
    inset = (rotation(center, angle - turn).to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y")) @ [*inset, 1])[:2]
    zoom_panel(ax_zoom, section, center, FRAME_SIZE, "mmp_cells", mmp_colors, "", point_size=0.5)
    ax_zoom.add_patch(Rectangle(inset - INSET_SIZE / 2, INSET_SIZE, INSET_SIZE, fill=False, ec=fs.YELLOW, lw=0.6))
    zoom_panel(ax_inset, section, inset, INSET_SIZE, "mmp_cells", mmp_colors, "", point_size=2)
    subfig.supxlabel(section, fontsize=7)
fig.legend(handles=[Patch(fc=color, ec="black", lw=0.3, label=name) for name, color in mmp_colors.items()]
           + [Line2D([], [], ls="", marker="o", ms=3, mew=0, color=color, label=fs.gene(gene)) for gene, color in GENE_COLORS.items()],
           loc="outside lower center", ncol=len(MMP_CELLS) + len(GENE_COLORS), frameon=False, handlelength=1, handleheight=0.8, fontsize=6, borderaxespad=0.2)
fig.get_layout_engine().set(h_pad=0.01)
save_panel(fig, "2e-f_zoom_inset")
plt.show()

In [ ]:
roi_map("cn_name", CN_COLORS, "2Sa_cn_map", "2Sa", "CN", fontsize=SUPP_PT, title_fontsize=SUPP_PT, wrap=True)

In [ ]:
fc = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "cn_celltype_enrichment.csv", index_col=0)

M = fc.iloc[leaves_list(linkage(fc.values, "average")), leaves_list(linkage(fc.values.T, "average"))]
M.index = [f"CN{c}" for c in M.index]
M = M.T
fig, ax = plt.subplots(figsize=panel_size("2Sb"), layout="constrained")
im = ax.imshow(M, cmap="PuOr_r", vmin=-3, vmax=3, aspect="auto", interpolation="none")
ax.set_yticks(range(len(M)), M.index)
ax.set_xticks(range(M.shape[1]), M.columns, rotation=90)
fig.colorbar(im, ax=ax, shrink=0.8, label="log$_2$ enrichment")
font7(fig)
save_panel(fig, "2Sb_cn_enrichment_heatmap")
plt.show()

In [ ]:
cn_counts = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "cn_composition_counts.csv", index_col=0)

ct = cn_counts
ct = ct.loc[:, ct.sum() > 0]
frac = ct.div(ct.sum(axis=1), axis=0) * 100
colors = [fs.CELL_COLORS.get(c, fs.LIGHTGREY) for c in ct.columns]
xs = np.arange(len(ct))
fig, ax = plt.subplots(figsize=panel_size("2Sc"), layout="constrained")
bottom = np.zeros(len(frac))
for c, color in zip(frac.columns, colors):
    ax.bar(xs, frac[c], bottom=bottom, width=0.8, color=color, edgecolor="white", linewidth=0.25, label=c)
    bottom += frac[c].to_numpy(dtype=float)
ax.set_xticks(xs, [f"CN{g}" for g in frac.index], rotation=90)
ax.set_ylabel("Proportion of cells (%)")
ax.set_ylim(0, 100)
handles, names = ax.get_legend_handles_labels()
fig.legend(handles[::-1], names[::-1], loc="outside right center", alignment="left",
           ncol=2, columnspacing=0.6, handlelength=0.8, handletextpad=0.3, labelspacing=0.3)
font7(fig)
save_panel(fig, "2Sc_cn_composition")
plt.show()

In [ ]:
where = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "celltype_by_cn_fraction.csv", index_col=0)
enr = pd.read_csv(RESULTS / "04_spatial_niche_analysis" / "celltype_by_cn_log2enrichment.csv", index_col=0)
LAYERS = ["Villus", "Crypt", "Muscularis", "Lymphoid aggregate"]
CN_LAYER = dict(zip(CN_COLORS, cn_info["layer"]))

cnames = list(where.columns)
markers = {layer: [n for n in cnames if CN_LAYER[n] == layer] for layer in LAYERS}
rows = list(where.index)
holder = ad.AnnData(np.zeros((len(rows), len(cnames)), dtype=np.float32),
                    obs=pd.DataFrame({"cell_type": pd.Categorical(rows, categories=rows)}, index=rows), var=pd.DataFrame(index=cnames))
def draw(width, height):
    dp = sc.pl.DotPlot(holder, markers, groupby="cell_type",
                       dot_color_df=enr.loc[rows, cnames], dot_size_df=where.loc[rows, cnames],
                       vmin=-3, vmax=3, vcenter=0,
                       figsize=(width, height))
    dp.swap_axes()
    dp.style(cmap="PuOr_r", dot_min=0, dot_max=1, largest_dot=40, dot_edge_lw=0.2)
    dp.legend(width=0.55, size_title="% cells", colorbar_title="log$_2$" + chr(10) + "enrichment")
    dp.make_figure()
    axes = dp.get_axes()
    size_ax, color_ax = axes.get("size_legend_ax"), axes.get("color_legend_ax")
    size_ax.set_xticklabels([t.get_text() if i % 2 == 0 else "" for i, t in enumerate(size_ax.get_xticklabels())])
    fig_w, fig_h = dp.fig.get_size_inches()
    p = axes["mainplot_ax"].get_position()
    leg_w = 0.6 / fig_w
    leg_y = p.y0 - 0.75 / fig_h
    color_ax.set_position([p.x0 - leg_w - 0.25 / fig_w, leg_y + 0.02 / fig_h, leg_w, 0.08 / fig_h])
    size_ax.set_position([p.x0 - 2 * leg_w - 0.55 / fig_w, leg_y, leg_w, 0.12 / fig_h])
    size_ax.set_ylim(-0.6, 0.6)
    size_ax.tick_params(axis="x", length=0, pad=1)
    size_ax.set_title(size_ax.get_title(), y=1.0, pad=2, size="small")
    right = 1 - 0.6 / fig_w
    g = axes["gene_group_ax"].get_position()
    axes["gene_group_ax"].set_position([right - g.width, g.y0, g.width, g.height])
    axes["mainplot_ax"].set_position([p.x0, p.y0, right - g.width - p.x0, p.height])
    for t, name in zip(axes["gene_group_ax"].texts, markers):
        t.set_text(name.replace(" ", chr(10)) if name == "Lymphoid aggregate" else name)
        t.set_rotation(0); t.set_horizontalalignment("left"); t.set_linespacing(1.0)
        t.set_verticalalignment("top" if name == "Lymphoid aggregate" else "center")
    axes["mainplot_ax"].set_ylabel("")
    color_ax.set_xticks([-3, 0, 3])
    return font7(dp.fig)

fig = fit_panel(draw, "2Sd")
save_panel(fig, "2Sd_cn_dotplot", tight=True)
plt.show()

In [ ]:
GENES = ["Mmp12", "Mmp13"]
sub = adata[(adata.obs["cell_subtype"].astype(str) != "Unassigned").values, GENES].copy()
names = sub.obs["cell_subtype"].astype(str)
sub.obs["cell_subtype"] = pd.Categorical(names, categories=sorted(names.unique()))
def draw(width, height):
    dp = sc.pl.dotplot(sub, GENES, groupby="cell_subtype", layer="log1p_norm", standard_scale="var",
                       dot_min=0, dot_max=0.5,
                       figsize=(width, height), show=False, return_fig=True)
    dp.style(largest_dot=40, dot_edge_lw=0.2)
    dp.legend(width=0.55, size_title="% cells", colorbar_title="Scaled\nexpression")
    dp.make_figure()
    tidy_dotplot(dp).set_ylabel("")
    size_ax = dp.get_axes()["size_legend_ax"]
    size_ax.set_xticklabels([t.get_text() if i in (0, len(size_ax.get_xticks()) - 1) else "" for i, t in enumerate(size_ax.get_xticklabels())])
    return font7(dp.fig)

fig = fit_panel(draw, "2Se")
save_panel(fig, "2Se_mmp_dotplot", tight=True)
plt.show()